In [15]:
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import os

In [16]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

In [17]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)
instruments_df.to_csv("NSE_Instruments.csv",index=False)

In [18]:
def instrumentLookup(instruments_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instruments_df[instruments_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1


def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data

In [19]:
def MACD(DF,a,b,c):
    """function to calculate MACD
       typical values a(fast moving average) = 12; 
                      b(slow moving average) =26; 
                      c(signal line ma window) =9"""
    df = DF.copy()
    df["MA_Fast"]=df["close"].ewm(span=a,min_periods=a).mean()
    df["MA_Slow"]=df["close"].ewm(span=b,min_periods=b).mean()
    df["MACD"]=df["MA_Fast"]-df["MA_Slow"]
    df["Signal"]=df["MACD"].ewm(span=c,min_periods=c).mean()
    df.dropna(inplace=True)
    return df

ohlc = fetchOHLC("ICICIBANK","5minute",5)
macd = MACD(ohlc,12,26,9)


In [20]:
macd

,open,high,low,close,volume,MA_Fast,MA_Slow,MACD,Signal
date,,,,,,,,,
2025-04-01 12:00:00+05:30,1327.15,1328.10,1324.55,1325.85,74137,1327.403218,1330.086436,-2.683219,-2.895924
2025-04-01 12:05:00+05:30,1325.85,1327.85,1325.65,1326.50,58542,1327.263858,1329.801503,-2.537645,-2.815649
2025-04-01 12:10:00+05:30,1326.45,1327.80,1325.45,1326.50,97873,1327.146054,1329.540609,-2.394555,-2.723516
2025-04-01 12:15:00+05:30,1326.50,1327.75,1326.00,1326.70,69340,1327.077288,1329.317241,-2.239953,-2.619667
2025-04-01 12:20:00+05:30,1326.60,1326.70,1324.25,1324.80,81157,1326.726322,1328.963646,-2.237325,-2.538750
...,...,...,...,...,...,...,...,...,...
2025-04-04 15:05:00+05:30,1334.50,1336.45,1334.40,1335.85,185687,1334.133160,1333.731657,0.401503,0.470019
2025-04-04 15:10:00+05:30,1335.70,1338.55,1335.10,1338.55,187375,1334.812674,1334.088571,0.724103,0.520836
2025-04-04 15:15:00+05:30,1338.40,1338.40,1335.00,1335.90,189060,1334.979955,1334.222751,0.757204,0.568109


In [21]:
ohlc

,open,high,low,close,volume
date,,,,,
2025-04-01 09:15:00+05:30,1340.00,1352.35,1339.55,1346.20,796452
2025-04-01 09:20:00+05:30,1345.90,1347.90,1344.05,1345.85,225771
2025-04-01 09:25:00+05:30,1345.90,1349.00,1341.65,1343.00,239011
2025-04-01 09:30:00+05:30,1342.40,1342.95,1337.00,1340.40,231862
2025-04-01 09:35:00+05:30,1340.65,1349.50,1338.80,1349.25,160504
...,...,...,...,...,...
2025-04-04 15:05:00+05:30,1334.50,1336.45,1334.40,1335.85,185687
2025-04-04 15:10:00+05:30,1335.70,1338.55,1335.10,1338.55,187375
2025-04-04 15:15:00+05:30,1338.40,1338.40,1335.00,1335.90,189060
